# Orbit Wars: Observation, Set Transformer ve Pair-Aware Target Head

Bu notebook'un amacı mevcut `Observation -> Set Transformer -> action heads` akışını sade şekilde açıklamak ve özellikle şu soruyu cevaplamak:

> Gezegenlerin kendi aralarındaki ilişkiyi öğrenmeye mevcut attention mekanizması yeter mi?

Kısa cevap: **evet, temel ilişki öğrenimi için yeterli**. Mevcut model gezegenleri set olarak görüyor, her gezegeni token yapıyor ve gezegenler arası geometriyi `edge_bias` ile attention skorlarına ekliyor. Fakat hedef seçimi (`src -> tgt`) sırasında edge bilgisi doğrudan skor fonksiyonuna girmediği için bazı geometrik kararları dolaylı öğreniyor. Bu yüzden yarışma için en pratik geliştirme, target head'e doğrudan `planet-planet edge` katkısı eklemek.

## 1. Mevcut Observation Neyi Taşıyor?

Observation üç ana gruptan oluşuyor:

- `planet_feats`: her gezegen için sahiplik, gemi sayısı, üretim, yarıçap, güneşe göre konum, incoming threat vb.
- `fleet_feats`: havadaki filolar için sahiplik, gemi sayısı, hız, yön, tahmini ETA vb.
- `edge_bias`: her token çifti için pairwise geometri.

Özellikle `edge_bias` kritik. Şekli:

```text
(T, T, EDGE_FEATURE_DIM)
T = P_max + F_max + 1
```

Token sırası:

```text
[planet_0, planet_1, ..., fleet_0, fleet_1, ..., GLOBAL]
```

Edge feature'ları:

```text
edge[i, j] = [
    distance(i, j),
    dx(i, j),
    dy(i, j),
    sin(angle_j - angle_i),
    cos(angle_j - angle_i),
]
```

Bu şu anlama geliyor: model sadece tek tek gezegenleri değil, gezegenlerin birbirine göre uzaklığını ve açısal ilişkisini de görüyor.

In [ ]:
# Bu hücre mevcut observation boyutlarını repo kodundan gösterir.
from obs_builder import (
    P_MAX_DEFAULT,
    F_MAX_DEFAULT,
    PLANET_FEATURE_DIM,
    FLEET_FEATURE_DIM,
    GLOBAL_FEATURE_DIM,
    EDGE_FEATURE_DIM,
)

T = P_MAX_DEFAULT + F_MAX_DEFAULT + 1
print("P_MAX:", P_MAX_DEFAULT)
print("F_MAX:", F_MAX_DEFAULT)
print("Token count T:", T)
print("planet_feats:", (P_MAX_DEFAULT, PLANET_FEATURE_DIM))
print("fleet_feats:", (F_MAX_DEFAULT, FLEET_FEATURE_DIM))
print("global_feats:", (GLOBAL_FEATURE_DIM,))
print("edge_bias:", (T, T, EDGE_FEATURE_DIM))

## 2. Set Transformer Bu İlişkiyi Nasıl Kullanıyor?

Modeldeki attention skoru kabaca şöyle:

```text
score(i, j) = q_i · k_j / sqrt(d) + edge_bias_projected(i, j)
```

Buradaki önemli nokta şu: `edge_bias` attention skoruna **additive bias** olarak giriyor. Yani model şunu öğrenebiliyor:

- Yakın gezegenlere daha fazla dikkat et.
- Aynı orbital hatta olan gezegenlere daha fazla dikkat et.
- Düşman tehdidi olan gezegenlerle kaynak gezegenleri beraber değerlendir.
- Global token üzerinden tüm harita dengesini özetle.

Bu yapı graph attention'a benzer: node'lar token, edge'ler pairwise geometri.

In [ ]:
import torch
import torch.nn as nn
import math

B, T, d_model, n_heads, edge_dim = 2, 8, 32, 4, 5
x = torch.randn(B, T, d_model)
edge_bias = torch.randn(B, T, T, edge_dim)

edge_proj = nn.Linear(edge_dim, n_heads)
projected_edge = edge_proj(edge_bias).permute(0, 3, 1, 2)  # (B, H, T, T)

q = torch.randn(B, n_heads, T, d_model // n_heads)
k = torch.randn(B, n_heads, T, d_model // n_heads)
content_score = q @ k.transpose(-2, -1) / math.sqrt(d_model // n_heads)

attention_score = content_score + projected_edge
print("content_score:", content_score.shape)
print("projected_edge:", projected_edge.shape)
print("attention_score:", attention_score.shape)

## 3. Mevcut Yapı Nerede Zayıf Kalıyor?

Encoder tarafında edge bilgisi kullanılıyor. Fakat target seçen head şu an sadece final gezegen embedding'lerini kullanıyor:

```python
q = self.tgt_query(planet_tok)
k = self.tgt_key(planet_tok)
tgt_logits = q @ k.transpose(-2, -1) / sqrt(d_model)
```

Bu şu demek:

- Distance/angle bilgisi target skoruna doğrudan girmiyor.
- Model bu bilgiyi encoder içinde gezegen embedding'lerine taşımak zorunda.
- Öğrenebilir, ama daha fazla veri ve eğitim stabilitesi ister.

Yani mevcut model `gezegen ilişkisi öğrenemez` değil. Daha doğru ifade:

> Mevcut model ilişki öğrenebilir, ama hedef seçimi sırasında pairwise geometriyi dolaylı kullanıyor.

## 4. Pair-Aware Target Head Fikri

Hedef seçimi tam olarak pairwise bir problem:

```text
Bu source gezegenden şu target gezegene saldırmak mantıklı mı?
```

Bu karar için sadece `src_token` ve `tgt_token` değil, doğrudan `edge(src, tgt)` de kullanılmalı.

Basit ve etkili formül:

```text
score(src, tgt)
= q(src) · k(tgt) / sqrt(d)
+ edge_target_proj(edge(src, tgt))
```

Bu versiyon hafif: sadece küçük bir `Linear(edge_dim, 1)` ekler.

Daha güçlü formül:

```text
score(src, tgt)
= q(src) · k(tgt) / sqrt(d)
+ MLP([src_token, tgt_token, edge(src, tgt)])
```

Yarışma için önce hafif versiyonu öneririm. Parametre az, davranış net, debug kolay.

In [ ]:
class LightweightPairAwareTargetHead(nn.Module):
    """Mevcut pointer target head + doğrudan planet-planet edge katkısı."""

    def __init__(self, d_model: int, edge_dim: int):
        super().__init__()
        self.tgt_query = nn.Linear(d_model, d_model)
        self.tgt_key = nn.Linear(d_model, d_model)
        self.edge_target_proj = nn.Linear(edge_dim, 1)
        self.d_model = d_model

    def forward(self, planet_tok, planet_edge):
        """
        planet_tok:  (B, P, d_model)
        planet_edge: (B, P, P, edge_dim)
        returns:     (B, P, P)
        """
        q = self.tgt_query(planet_tok)
        k = self.tgt_key(planet_tok)
        pointer_score = q @ k.transpose(-2, -1) / math.sqrt(self.d_model)
        edge_score = self.edge_target_proj(planet_edge).squeeze(-1)
        return pointer_score + edge_score


B, P, d_model, edge_dim = 2, 48, 128, 5
planet_tok = torch.randn(B, P, d_model)
planet_edge = torch.randn(B, P, P, edge_dim)

head = LightweightPairAwareTargetHead(d_model=d_model, edge_dim=edge_dim)
tgt_logits = head(planet_tok, planet_edge)
print(tgt_logits.shape)

## 5. Mevcut `OrbitWarsPolicy` İçinde Nereye Eklenir?

Mevcut `forward` içinde target logits şu bölümde hesaplanıyor:

```python
q = self.tgt_query(planet_tok)
k = self.tgt_key(planet_tok)
tgt_logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)
```

Hafif pair-aware versiyonda `__init__` içine bir layer eklenir:

```python
self.edge_target_proj = nn.Linear(edge_dim, 1)
```

`forward` içinde planet-planet edge alınır:

```python
planet_edge = edge_bias[:, :self.p_max, :self.p_max]
edge_score = self.edge_target_proj(planet_edge).squeeze(-1)
tgt_logits = tgt_logits + edge_score
```

Sonra mevcut maskeler aynen kalır:

```python
tgt_logits = tgt_logits.masked_fill(eye[None], very_neg)
tgt_logits = tgt_logits.masked_fill(tgt_pad[:, None, :], very_neg)
```

Bu değişiklik action API'yi bozmaz. `tgt_logits` şekli yine `(B, P, P)` kalır.

In [ ]:
# Mevcut forward içine yapılacak minimal değişikliğin izole örneği.
B, P, d_model, edge_dim = 2, 4, 16, 5
planet_tok = torch.randn(B, P, d_model)
edge_bias = torch.randn(B, P, P, edge_dim)
planet_mask = torch.tensor([[True, True, True, False], [True, True, True, True]])

tgt_query = nn.Linear(d_model, d_model)
tgt_key = nn.Linear(d_model, d_model)
edge_target_proj = nn.Linear(edge_dim, 1)

q = tgt_query(planet_tok)
k = tgt_key(planet_tok)
tgt_logits = q @ k.transpose(-2, -1) / math.sqrt(d_model)
tgt_logits = tgt_logits + edge_target_proj(edge_bias).squeeze(-1)

very_neg = torch.finfo(tgt_logits.dtype).min
eye = torch.eye(P, dtype=torch.bool)
tgt_logits = tgt_logits.masked_fill(eye[None], very_neg)
tgt_logits = tgt_logits.masked_fill((~planet_mask)[:, None, :], very_neg)

print(tgt_logits.shape)
print(tgt_logits[0])

## 6. Neden Manifold Algoritması Değil?

Isomap, LLE, t-SNE ve UMAP gibi yöntemler genelde şu işler için daha uygun:

- yüksek boyutlu veriyi 2D/3D görselleştirme,
- offline embedding üretme,
- veri dağılımını analiz etme.

Orbit Wars ise dinamik, maskeli, aksiyon odaklı bir karar problemi. Her turn'de sahiplik, gemi sayısı, incoming fleet ve hedef önceliği değişiyor. Bu yüzden offline manifold embedding yerine, modelin karar anında pairwise geometriyi kullanması daha doğru.

Set Transformer + edge bias burada zaten manifold fikrine benzer bir şeyi öğrenebilir: yakınlık, açısal komşuluk ve stratejik bağlamı attention üzerinden kullanır. Eksik parça, bu pairwise sinyali target head'e daha doğrudan vermek.

## 7. Pratik Karar

Mevcut model için değerlendirme:

```text
Stratejik ilişki öğrenme: yeterli
Yakın/uzak, değerli/değersiz, tehdit altında, sahiplik dengesi: yeterli
Hedef seçimi için kaba geometrik ilişki: yeterli
Kesin launch kalitesi, ETA, interception, future position: zayıf/örtük
```

Önerilen sıra:

1. Önce mevcut modeli baseline olarak çalıştır.
2. Sonra sadece `edge_target_proj` eklenmiş pair-aware target head'i dene.
3. Aynı seed ve aynı eğitim ayarlarıyla karşılaştır:
   - ortalama oyun uzunluğu,
   - launch sayısı,
   - win/draw oranı,
   - invalid launch oranı,
   - target head gradient/update ratio.

Bu değişiklik küçük olduğu için yarışma riskini düşük tutar. Modelin aksiyon API'si değişmez, checkpoint uyumsuzluğu dışında rollout/ppo kodunun çoğu aynı kalır.